In [ ]:
!pip install -q transformers torch chromadb PyPDF2 bitsandbytes accelerate sentencepiece protobuf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 9.9 MB/s eta 0:

In [ ]:
!pip install rank_bm25

In [ ]:
"""
Legal RAG System - FINAL VERSION WITH HYBRID KEYWORD + SEMANTIC SEARCH
Fixes poor retrieval by combining BM25 keyword search with semantic search
"""

import os
import json
import gc
import re
import torch
import numpy as np
import chromadb
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from typing import List, Dict, Optional, Tuple
import shutil
import warnings
from collections import Counter
warnings.filterwarnings('ignore')


# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration settings"""

    EMBEDDING_MODEL = "nlpaueb/legal-bert-base-uncased"
    LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

    USE_4BIT_QUANTIZATION = True

    PERSIST_DIRECTORY = "./legal_vector_db"
    COLLECTION_NAME = "legal_statutes"

    MAX_NEW_TOKENS = 2048  # Increased for better answers
    TEMPERATURE = 0.7
    TOP_P = 0.9

    EMBEDDING_BATCH_SIZE = 32
    SECTION_SEPARATOR = "=" * 80

    # Retrieval settings
    SEMANTIC_TOP_K = 10  # Retrieve more candidates
    FINAL_TOP_K = 5      # Return top 5 after reranking
    BM25_WEIGHT = 0.4    # Weight for keyword search
    SEMANTIC_WEIGHT = 0.6  # Weight for semantic search


# ============================================================================
# BM25 KEYWORD SEARCH
# ============================================================================

class BM25:
    """Simple BM25 implementation for keyword search"""

    def __init__(self, corpus: List[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.corpus_size = len(corpus)

        # Tokenize corpus
        self.tokenized_corpus = [self.tokenize(doc) for doc in corpus]

        # Calculate document frequencies
        self.doc_freqs = self._calculate_doc_freqs()

        # Calculate IDF
        self.idf = self._calculate_idf()

        # Calculate average document length
        self.avgdl = sum(len(doc) for doc in self.tokenized_corpus) / self.corpus_size

    def tokenize(self, text: str) -> List[str]:
        """Simple tokenization - lowercase and split"""
        # Remove punctuation and lowercase
        text = re.sub(r'[^\w\s]', ' ', text.lower())
        return text.split()

    def _calculate_doc_freqs(self) -> Dict[str, int]:
        """Calculate how many documents each term appears in"""
        doc_freqs = {}
        for doc in self.tokenized_corpus:
            unique_terms = set(doc)
            for term in unique_terms:
                doc_freqs[term] = doc_freqs.get(term, 0) + 1
        return doc_freqs

    def _calculate_idf(self) -> Dict[str, float]:
        """Calculate IDF for each term"""
        idf = {}
        for term, freq in self.doc_freqs.items():
            idf[term] = np.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1)
        return idf

    def get_scores(self, query: str) -> np.ndarray:
        """Calculate BM25 scores for query against all documents"""
        query_tokens = self.tokenize(query)
        scores = np.zeros(self.corpus_size)

        for doc_idx, doc in enumerate(self.tokenized_corpus):
            doc_len = len(doc)
            doc_term_freqs = Counter(doc)

            score = 0
            for term in query_tokens:
                if term not in self.idf:
                    continue

                term_freq = doc_term_freqs.get(term, 0)
                idf = self.idf[term]

                # BM25 formula
                numerator = term_freq * (self.k1 + 1)
                denominator = term_freq + self.k1 * (1 - self.b + self.b * (doc_len / self.avgdl))
                score += idf * (numerator / denominator)

            scores[doc_idx] = score

        return scores


# ============================================================================
# QUERY PARSER
# ============================================================================

class QueryParser:
    """Parses queries to detect exact section requests and extract key terms"""

    @staticmethod
    def extract_section_ids(query: str) -> List[str]:
        """Extract section IDs from query"""
        section_ids = []

        # Pattern 1: "section 514", "sec 514", "s 514"
        pattern1 = r'\b(?:section|sec|s\.?)\s+(\d+)'
        matches = re.finditer(pattern1, query, re.IGNORECASE)
        for match in matches:
            section_num = match.group(1)
            section_ids.append(f"section_{section_num}")

        # Pattern 2: "CA_514" style
        pattern2 = r'\b([A-Z]+_\d+)\b'
        matches = re.finditer(pattern2, query)
        for match in matches:
            section_ids.append(match.group(1))

        # Pattern 3: Just a number if query is very short
        if len(query.strip()) < 10 and query.strip().isdigit():
            section_ids.append(f"section_{query.strip()}")

        return list(set(section_ids))

    @staticmethod
    def is_exact_section_query(query: str) -> bool:
        """Check if query is asking for specific section(s)"""
        patterns = [
            r'\bsection\s+\d+',
            r'\bsec\s+\d+',
            r'\bs\.\s*\d+',
            r'\b[A-Z]+_\d+\b',
        ]

        for pattern in patterns:
            if re.search(pattern, query, re.IGNORECASE):
                return True

        if len(query.strip()) < 10 and query.strip().isdigit():
            return True

        return False

    @staticmethod
    def extract_legal_terms(query: str) -> List[str]:
        """
        Extract key legal terms from query for better keyword matching
        """
        # Common legal terms to look for
        legal_keywords = [
            'allotment', 'share', 'shares', 'director', 'resolution', 'special resolution',
            'register', 'member', 'shareholders', 'certificate', 'registrar', 'return',
            'pre-emption', 'preemption', 'proportion', 'existing members', 'dilution',
            'penalty', 'breach', 'contract', 'compensation', 'liquidator', 'winding up',
            'company', 'private company', 'public company', 'articles', 'memorandum',
            'board', 'meeting', 'general meeting', 'extraordinary', 'ordinary',
            'dividend', 'capital', 'debenture', 'secured', 'unsecured', 'creditor',
            'amalgamation', 'merger', 'takeover', 'tribunal', 'court', 'application'
        ]

        query_lower = query.lower()
        found_terms = []

        for term in legal_keywords:
            if term in query_lower:
                found_terms.append(term)

        return found_terms


# ============================================================================
# SECTION LOADER
# ============================================================================

class SectionLoader:
    """Loads sections from text file and metadata from JSON"""

    @staticmethod
    def load_sections_from_file(txt_path: str) -> List[str]:
        """Load sections from text file"""
        print(f"📄 Loading sections from: {os.path.basename(txt_path)}")

        with open(txt_path, 'r', encoding='utf-8') as f:
            content = f.read()

        sections = content.split(Config.SECTION_SEPARATOR)

        cleaned_sections = []
        for section in sections:
            section = section.strip()
            if section and len(section) > 50:
                cleaned_sections.append(section)

        print(f"   ✓ Loaded {len(cleaned_sections)} sections")
        return cleaned_sections

    @staticmethod
    def process_metadata_structure(data, depth=0) -> List[Dict]:
        """Process any metadata structure recursively"""
        if depth > 5:
            return []

        metadata = []

        if isinstance(data, list):
            if len(data) == 0:
                return []

            first_item = data[0]

            if isinstance(first_item, dict):
                print(f"   ✓ Found list of {len(data)} dictionaries")
                metadata = data

            elif isinstance(first_item, str):
                print(f"   ⚠️  Found list of {len(data)} strings - creating basic metadata")
                for i, item in enumerate(data):
                    metadata.append({
                        'section_id': f'section_{i}',
                        'act': 'Unknown Act',
                        'title': item[:100] if len(item) > 100 else item,
                    })

            elif isinstance(first_item, list):
                print(f"   ⚠️  Found nested lists - flattening...")
                for sublist in data:
                    metadata.extend(SectionLoader.process_metadata_structure(sublist, depth + 1))

        elif isinstance(data, dict):
            print(f"   ✓ Found dictionary with {len(data)} keys")

            if 'sections' in data:
                print(f"   ✓ Found 'sections' key - extracting...")
                return SectionLoader.process_metadata_structure(data['sections'], depth + 1)

            elif 'documents' in data:
                print(f"   ✓ Found 'documents' key - extracting...")
                return SectionLoader.process_metadata_structure(data['documents'], depth + 1)

            elif 'data' in data:
                print(f"   ✓ Found 'data' key - extracting...")
                return SectionLoader.process_metadata_structure(data['data'], depth + 1)

            else:
                print(f"   ✓ Treating as section_id -> metadata mapping")
                for section_id, section_data in data.items():
                    if isinstance(section_data, dict):
                        section_data['section_id'] = section_id
                        metadata.append(section_data)
                    else:
                        metadata.append({
                            'section_id': section_id,
                            'act': 'Unknown Act',
                            'title': str(section_data)[:100],
                        })

        return metadata

    @staticmethod
    def load_metadata_from_json(json_path: str) -> List[Dict]:
        """Load metadata from JSON file"""
        print(f"📋 Loading metadata from: {os.path.basename(json_path)}")

        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        print(f"\n   🔍 JSON top-level type: {type(raw_data).__name__}")

        metadata = SectionLoader.process_metadata_structure(raw_data)

        if not metadata:
            print(f"   ❌ Could not extract metadata!")
            return []

        print(f"   ✓ Loaded metadata for {len(metadata)} sections")

        if metadata and len(metadata) > 0:
            print(f"\n   📄 Sample metadata:")
            sample = metadata[0]
            for key, value in list(sample.items())[:5]:
                if isinstance(value, str) and len(value) > 100:
                    print(f"      {key}: {value[:100]}...")
                else:
                    print(f"      {key}: {value}")

        return metadata

    @staticmethod
    def match_sections_with_metadata(sections: List[str], metadata: List[Dict]) -> List[Dict]:
        """Match sections with their metadata"""
        print(f"\n🔗 Matching sections with metadata...")
        print(f"   Sections: {len(sections)}")
        print(f"   Metadata entries: {len(metadata)}")

        matched_documents = []

        if metadata and not isinstance(metadata[0], dict):
            print(f"   ⚠️  Metadata items are {type(metadata[0])}, not dict!")
            metadata = [
                {
                    'section_id': f'section_{i}',
                    'act': 'Unknown Act',
                    'title': f'Section {i}',
                }
                for i in range(len(sections))
            ]

        if len(sections) == len(metadata):
            print("   ✓ Counts match - pairing by order")
            for i, (section, meta) in enumerate(zip(sections, metadata)):
                matched_documents.append({
                    'text': section,
                    'section_id': meta.get('section_id', f'section_{i}'),
                    'act': meta.get('act', 'Unknown Act'),
                    'title': meta.get('title', 'No Title'),
                    'metadata': meta
                })

        else:
            print(f"   ⚠️  Count mismatch - using first {min(len(sections), len(metadata))}")

            for i in range(len(sections)):
                if i < len(metadata):
                    meta = metadata[i]
                    matched_documents.append({
                        'text': sections[i],
                        'section_id': meta.get('section_id', f'section_{i}'),
                        'act': meta.get('act', 'Unknown Act'),
                        'title': meta.get('title', 'No Title'),
                        'metadata': meta
                    })
                else:
                    matched_documents.append({
                        'text': sections[i],
                        'section_id': f'section_{i}',
                        'act': 'Unknown Act',
                        'title': sections[i][:50] + '...',
                        'metadata': {}
                    })

        print(f"   ✓ Matched {len(matched_documents)} documents")

        # Check for duplicates
        id_counts = {}
        for doc in matched_documents:
            sid = doc['section_id']
            id_counts[sid] = id_counts.get(sid, 0) + 1

        duplicates = {id: count for id, count in id_counts.items() if count > 1}
        if duplicates:
            print(f"\n   ⚠️  WARNING: Found {len(duplicates)} duplicate section IDs!")
            print(f"   First 3 duplicates:")
            for dup_id, count in list(duplicates.items())[:3]:
                print(f"      {dup_id}: appears {count} times")

            response = input("\n   Auto-fix by adding suffixes? (yes/no): ")
            if response.lower() in ['yes', 'y']:
                print(f"   🔧 Auto-fixing...")
                seen_ids = {}
                for doc in matched_documents:
                    original_id = doc['section_id']
                    if original_id in seen_ids:
                        seen_ids[original_id] += 1
                        new_id = f"{original_id}_v{seen_ids[original_id]}"
                        doc['section_id'] = new_id
                    else:
                        seen_ids[original_id] = 0
                print(f"   ✓ Fixed")

        if matched_documents:
            sample = matched_documents[0]
            print(f"\n   📄 Sample:")
            print(f"      ID: {sample['section_id']}")
            print(f"      Title: {sample['title'][:60]}...")

        return matched_documents


# ============================================================================
# HYBRID RAG SYSTEM
# ============================================================================

class HybridLegalRAG:
    """Legal RAG with BM25 keyword + semantic search"""

    def __init__(self,
                 embedding_model_name: str = Config.EMBEDDING_MODEL,
                 llm_model_name: str = Config.LLM_MODEL,
                 persist_directory: str = Config.PERSIST_DIRECTORY,
                 use_quantization: bool = Config.USE_4BIT_QUANTIZATION):

        print("="*80)
        print("HYBRID LEGAL RAG - KEYWORD + SEMANTIC SEARCH")
        print("="*80)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Device: {self.device}")

        print(f"\n1️⃣  Loading embedding model...")
        self.embedding_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
        self.embedding_model = AutoModel.from_pretrained(embedding_model_name).to(self.device)
        self.embedding_model.eval()
        print("   ✓ Loaded")

        print(f"\n2️⃣  Setting up vector database...")
        os.makedirs(persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=persist_directory)

        try:
            self.collection = self.client.get_collection(Config.COLLECTION_NAME)
            print(f"   ✓ Loaded existing collection ({self.collection.count()} docs)")
        except:
            self.collection = self.client.create_collection(
                name=Config.COLLECTION_NAME,
                metadata={"hnsw:space": "cosine"}
            )
            print("   ✓ Created new collection")

        print(f"\n3️⃣  Loading LLM...")

        self.llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

        if self.llm_tokenizer.pad_token is None:
            self.llm_tokenizer.pad_token = self.llm_tokenizer.eos_token

        if use_quantization and self.device == "cuda":
            print("   (Using 4-bit quantization...)")
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4"
            )

            self.llm_model = AutoModelForCausalLM.from_pretrained(
                llm_model_name,
                quantization_config=quantization_config,
                device_map="auto",
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )
        else:
            self.llm_model = AutoModelForCausalLM.from_pretrained(
                llm_model_name,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                device_map="auto" if self.device == "cuda" else None,
                trust_remote_code=True,
                low_cpu_mem_usage=True
            )
            if self.device == "cpu":
                self.llm_model = self.llm_model.to(self.device)

        self.llm_model.eval()
        print("   ✓ Loaded")

        # BM25 index (will be built during ingestion)
        self.bm25 = None
        self.documents_text = []
        self.documents_metadata = []

        self.conversation_history = []
        self.max_history = 5

        print("\n✅ System ready!")
        print("="*80)

    def get_embeddings(self, texts: List[str], batch_size: int = Config.EMBEDDING_BATCH_SIZE) -> np.ndarray:
        """Generate embeddings"""
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            inputs = self.embedding_tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                max_length=512,
                padding=True
            ).to(self.device)

            with torch.no_grad():
                outputs = self.embedding_model(**inputs)
                embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                all_embeddings.extend(embeddings)

            del inputs, outputs
            if self.device == "cuda":
                torch.cuda.empty_cache()

        return np.array(all_embeddings)

    def ingest_documents(self, sections_txt_path: str, metadata_json_path: str):
        """Ingest documents and build BM25 index"""
        print("="*80)
        print("DOCUMENT INGESTION")
        print("="*80)

        sections = SectionLoader.load_sections_from_file(sections_txt_path)
        metadata_list = SectionLoader.load_metadata_from_json(metadata_json_path)
        documents = SectionLoader.match_sections_with_metadata(sections, metadata_list)

        if not documents:
            print("\n❌ No documents!")
            return

        print(f"\n🔮 Generating embeddings for {len(documents)} sections...")
        searchable_texts = [
            f"{doc['section_id']}: {doc['title']}. {doc['text']}"
            for doc in documents
        ]

        embeddings = self.get_embeddings(searchable_texts)

        print(f"\n💾 Storing in vector database...")

        ids = [doc['section_id'] for doc in documents]
        texts = [doc['text'] for doc in documents]
        metadatas = [
            {
                'section_id': doc['section_id'],
                'act': doc['act'],
                'title': doc['title'],
                'word_count': len(doc['text'].split()),
                'citation': doc['section_id']
            }
            for doc in documents
        ]

        # Check for duplicates
        if len(ids) != len(set(ids)):
            duplicates = [id for id in ids if ids.count(id) > 1]
            unique_dups = list(set(duplicates))
            print(f"\n   ❌ ERROR: Found {len(unique_dups)} duplicate IDs!")
            for dup in unique_dups[:5]:
                print(f"      {dup}: appears {ids.count(dup)} times")
            print(f"\n   Fix your data or use auto-fix during matching!")
            return

        # Add to ChromaDB
        batch_size = 100
        for i in range(0, len(ids), batch_size):
            batch_end = min(i + batch_size, len(ids))
            self.collection.add(
                embeddings=embeddings[i:batch_end].tolist(),
                documents=texts[i:batch_end],
                metadatas=metadatas[i:batch_end],
                ids=ids[i:batch_end]
            )

        # Build BM25 index
        print(f"\n🔍 Building BM25 keyword index...")
        self.documents_text = texts
        self.documents_metadata = metadatas
        self.bm25 = BM25(texts)
        print(f"   ✓ BM25 index built")

        print(f"\n✅ Ingested {len(documents)} sections!")
        print(f"   Total in DB: {self.collection.count()}")
        print("="*80)

    def retrieve_exact(self, section_ids: List[str]) -> List[Dict]:
        """Retrieve exact sections by ID"""
        print(f"   🎯 Exact lookup for: {', '.join(section_ids)}")

        retrieved_docs = []

        for section_id in section_ids:
            try:
                result = self.collection.get(
                    ids=[section_id],
                    include=['documents', 'metadatas']
                )

                if result['documents'] and len(result['documents']) > 0:
                    retrieved_docs.append({
                        'text': result['documents'][0],
                        'metadata': result['metadatas'][0],
                        'similarity': 1.0,
                        'match_type': 'exact',
                        'score': 1.0
                    })
                    print(f"      ✓ Found {section_id}")
                else:
                    print(f"      ✗ Not found: {section_id}")
            except Exception as e:
                print(f"      ✗ Error: {e}")

        return retrieved_docs

    def retrieve_hybrid(self, query: str, top_k: int = Config.SEMANTIC_TOP_K) -> List[Dict]:
        """
        Hybrid retrieval: BM25 keyword + semantic search
        """
        if not self.bm25:
            print("   ⚠️  BM25 not initialized, using semantic only")
            return self.retrieve_semantic(query, top_k)

        print(f"   🔍 Hybrid search (BM25 + semantic)...")

        # 1. BM25 keyword scores
        bm25_scores = self.bm25.get_scores(query)
        bm25_scores_norm = bm25_scores / (bm25_scores.max() + 1e-6)  # Normalize

        # 2. Semantic scores
        query_embedding = self.get_embeddings([query])[0]
        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(top_k, self.collection.count())
        )

        # Create semantic score mapping by section_id
        semantic_scores = {}
        if results['metadatas'] and len(results['metadatas'][0]) > 0:
            for i, meta in enumerate(results['metadatas'][0]):
                section_id = meta['section_id']
                distance = results['distances'][0][i]
                similarity = max(0, min(1, 1 - (distance / 2)))
                semantic_scores[section_id] = similarity

        # 3. Combine scores
        combined_results = []

        for idx, (text, meta) in enumerate(zip(self.documents_text, self.documents_metadata)):
            section_id = meta['section_id']

            # Get scores
            bm25_score = bm25_scores_norm[idx]
            semantic_score = semantic_scores.get(section_id, 0.0)

            # Weighted combination
            combined_score = (
                Config.BM25_WEIGHT * bm25_score +
                Config.SEMANTIC_WEIGHT * semantic_score
            )

            if combined_score > 0:  # Only include if some relevance
                combined_results.append({
                    'text': text,
                    'metadata': meta,
                    'similarity': semantic_score,
                    'bm25_score': float(bm25_score),
                    'combined_score': float(combined_score),
                    'match_type': 'hybrid'
                })

        # Sort by combined score
        combined_results.sort(key=lambda x: x['combined_score'], reverse=True)

        # Return top results
        return combined_results[:Config.FINAL_TOP_K]

    def retrieve_semantic(self, query: str, top_k: int = 3) -> List[Dict]:
        """Semantic search only (fallback)"""
        count = self.collection.count()

        if count == 0:
            return []

        query_embedding = self.get_embeddings([query])[0]

        results = self.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(top_k, count)
        )

        retrieved_docs = []
        if results['documents'] and len(results['documents'][0]) > 0:
            for i in range(len(results['documents'][0])):
                distance = results['distances'][0][i]
                similarity = max(0, min(1, 1 - (distance / 2)))

                retrieved_docs.append({
                    'text': results['documents'][0][i],
                    'metadata': results['metadatas'][0][i],
                    'similarity': similarity,
                    'match_type': 'semantic',
                    'score': similarity
                })

        return retrieved_docs

    def retrieve(self, query: str, top_k: int = Config.FINAL_TOP_K) -> Tuple[List[Dict], str]:
        """
        Smart retrieval: exact, hybrid, or semantic
        """
        # Check if exact section query
        if QueryParser.is_exact_section_query(query):
            section_ids = QueryParser.extract_section_ids(query)

            if section_ids:
                exact_docs = self.retrieve_exact(section_ids)

                if exact_docs:
                    return exact_docs, 'exact'
                else:
                    print(f"   ⚠️  Exact sections not found, using hybrid search...")

        # Use hybrid search for general queries
        hybrid_docs = self.retrieve_hybrid(query, Config.SEMANTIC_TOP_K)
        return hybrid_docs, 'hybrid'

    def generate_answer(self, query: str, context: str, citations: List[str]) -> str:
        """Generate answer"""

        citations_text = "\n".join([f"- {cite}" for cite in citations])

        prompt = f"""<s>[INST] You are a legal expert. Answer based ONLY on the provided legal text.

Legal Text from:
{citations_text}

Full Legal Text:
{context}

Question: {query}

Instructions:
1. Answer based ONLY on the provided legal text
2. Cite specific sections when making statements (e.g., "According to CA_80...")
3. Be precise and accurate
4. If information is not in the provided text, say so clearly
5. For complex scenarios, identify ALL relevant sections

Answer: [/INST]"""

        inputs = self.llm_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=3000,  # Longer context
            padding=True
        )

        if hasattr(self.llm_model, 'device'):
            inputs = inputs.to(self.llm_model.device)
        else:
            inputs = inputs.to(self.device)

        print("⏳ Generating...")

        with torch.no_grad():
            outputs = self.llm_model.generate(
                **inputs,
                max_new_tokens=Config.MAX_NEW_TOKENS,
                temperature=Config.TEMPERATURE,
                top_p=Config.TOP_P,
                do_sample=True,
                pad_token_id=self.llm_tokenizer.pad_token_id,
                eos_token_id=self.llm_tokenizer.eos_token_id
            )

        response = self.llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "[/INST]" in response:
            answer = response.split("[/INST]")[-1].strip()
        else:
            answer = response

        del inputs, outputs
        if self.device == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

        return answer

    def query(self, question: str, top_k: int = Config.FINAL_TOP_K) -> Dict:
        """Query with hybrid search"""
        print(f"\n{'='*80}")
        print(f"❓ {question[:100]}..." if len(question) > 100 else f"❓ {question}")
        print(f"{'='*80}")

        print(f"\n📊 Searching {self.collection.count()} sections...")

        # Extract legal terms for better display
        legal_terms = QueryParser.extract_legal_terms(question)
        if legal_terms:
            print(f"   🔑 Key terms: {', '.join(legal_terms[:5])}")

        retrieved_docs, search_type = self.retrieve(question, top_k)

        if not retrieved_docs:
            print("\n⚠️  No sections found!")
            return {"answer": "No information found.", "sources": [], "citations": []}

        print(f"✓ Retrieved {len(retrieved_docs)} sections ({search_type} match)\n")

        print("="*80)
        print("RETRIEVED:")
        print("="*80)

        for i, doc in enumerate(retrieved_docs, 1):
            meta = doc['metadata']

            if search_type == 'exact':
                match_indicator = "🎯 EXACT"
                score_str = f"Match: 1.000"
            elif search_type == 'hybrid':
                match_indicator = "🔍 Hybrid"
                score_str = f"Combined: {doc['combined_score']:.3f} (BM25: {doc['bm25_score']:.3f}, Semantic: {doc['similarity']:.3f})"
            else:
                match_indicator = "🔍 Semantic"
                score_str = f"Similarity: {doc['similarity']:.3f}"

            print(f"\n[{i}] {match_indicator}: {meta['section_id']}")
            print(f"    Title: {meta['title']}")
            print(f"    {score_str}")
            print(f"    {doc['text'][:200]}...")
            print("-" * 40)

        context = "\n\n".join([doc['text'] for doc in retrieved_docs])
        citations = [f"{doc['metadata']['section_id']}: {doc['metadata']['title']}"
                    for doc in retrieved_docs]

        answer = self.generate_answer(question, context, citations)

        print(f"\n{'='*80}")
        print("ANSWER:")
        print("="*80)
        print(answer)
        print("="*80)

        return {
            "answer": answer,
            "sources": retrieved_docs,
            "citations": citations,
            "search_type": search_type
        }


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("\n" + "="*80)
    print("HYBRID LEGAL RAG - BM25 + SEMANTIC SEARCH")
    print("="*80)
    print("Features:")
    print("  🔑 BM25 keyword search (finds exact terms)")
    print("  🧠 Semantic search (understands meaning)")
    print("  🎯 Exact section lookup (section 514 → returns section 514)")
    print("  🔄 Hybrid scoring (combines keyword + semantic)")
    print("="*80)

    sections_txt_path = "/content/sections.txt"
    metadata_json_path = "/content/sections_metadata.json"

    if not os.path.exists(sections_txt_path):
        print(f"\n❌ Not found: {sections_txt_path}")
        return

    if not os.path.exists(metadata_json_path):
        print(f"\n❌ Not found: {metadata_json_path}")
        return

    try:
        rag = HybridLegalRAG()
    except Exception as e:
        print(f"\n❌ Init error: {e}")
        import traceback
        traceback.print_exc()
        return

    if rag.collection.count() == 0:
        print("\n📥 Ingesting...")
        try:
            rag.ingest_documents(sections_txt_path, metadata_json_path)
        except Exception as e:
            print(f"\n❌ Ingestion error: {e}")
            import traceback
            traceback.print_exc()
            return
    else:
        print(f"\n✓ Database has {rag.collection.count()} documents")
        response = input("Re-ingest? (yes/no): ")
        if response.lower() in ['yes', 'y']:
            if os.path.exists(Config.PERSIST_DIRECTORY):
                shutil.rmtree(Config.PERSIST_DIRECTORY)
            rag = HybridLegalRAG()
            rag.ingest_documents(sections_txt_path, metadata_json_path)

    print("\n" + "="*80)
    print("🎯 READY")
    print("="*80)
    print("Try:")
    print("  - Complex scenario (will use hybrid search)")
    print("  - 'section 514' (exact lookup)")
    print("  - 'penalties for breach' (keyword + semantic)")
    print("="*80)

    while True:
        print("\nQuestion (or 'quit'):")
        try:
            user_input = input("> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Bye!")
            break

        if not user_input or user_input.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Bye!")
            break

        try:
            rag.query(user_input)
        except Exception as e:
            print(f"\n❌ Error: {e}")
            import traceback
            traceback.print_exc()


if __name__ == "__main__":
    main()


HYBRID LEGAL RAG - BM25 + SEMANTIC SEARCH
Features:
  🔑 BM25 keyword search (finds exact terms)
  🧠 Semantic search (understands meaning)
  🎯 Exact section lookup (section 514 → returns section 514)
  🔄 Hybrid scoring (combines keyword + semantic)
HYBRID LEGAL RAG - KEYWORD + SEMANTIC SEARCH
Device: cuda

1️⃣  Loading embedding model...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

   ✓ Loaded

✅ System ready!

📥 Ingesting...
DOCUMENT INGESTION
📄 Loading sections from: sections.txt
   ✓ Loaded 507 sections
📋 Loading metadata from: sections_metadata.json

   🔍 JSON top-level type: dict
   ✓ Found dictionary with 3 keys
   ✓ Found 'sections' key - extracting...
   ✓ Found list of 507 dictionaries
   ✓ Loaded metadata for 507 sections

   📄 Sample metadata:
      section_id: CA_1
      section_number: 1
      title: Short title, extent and commencement.
      act: Companies Act, 2017
      text: 1. Short title, extent and commencement..—
(1) This Act may be
called the Companies Act, 2017.
(2) I...

🔗 Matching sections with metadata...
   Sections: 507
   Metadata entries: 507
   ✓ Counts match - pairing by order
   ✓ Matched 507 documents

   ⚠️  WARNING: Found 21 duplicate section IDs!
   First 3 duplicates:
      CA_1: appears 6 times
      CA_2: appears 6 times
      CA_5: appears 2 times

   Auto-fix by adding suffixes? (yes/no): yes
   🔧 Auto-fixing...
   ✓ Fix